In [ ]:
import ee
import geemap

ee.Authenticate()
ee.Initialize(project='fir-lab16-f14ef')

: 

In [ ]:
regions = ee.FeatureCollection('FAO/GAUL/2015/level1')
rostov = regions.filter(ee.Filter.eq('ADM1_NAME', 'Rostovskaya Oblast'))

In [ ]:
crops = ee.ImageCollection("ESA/WorldCereal/2021/MODELS/v100")

winter = crops.filter(ee.Filter.eq('product', 'wintercereals')).mosaic().select('classification')
spring = crops.filter(ee.Filter.eq('product', 'springcereals')).mosaic().select('classification')
maize = crops.filter(ee.Filter.eq('product', 'maize')).mosaic().select('classification')

Map = geemap.Map(basemap='SATELLITE')
Map.centerObject(rostov, 7)
Map.addLayer(spring.updateMask(spring.gt(50)).clip(rostov), {'palette': ['red']}, 'Яровые')
Map.addLayer(winter.updateMask(winter.gt(50)).clip(rostov), {'palette': ['orange']}, 'Озимые')
Map.addLayer(maize.updateMask(maize.gt(50)).clip(rostov), {'palette': ['green']}, 'Кукуруза')
Map.addLayer(rostov.style(color='white', fillColor='00000000'), {}, 'Граница')
Map

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

ndvi = ee.ImageCollection('MODIS/061/MOD13Q1') # индекс растительности
era5 = ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR') # климат

region = rostov.geometry()

In [ ]:
def get_data(mask, ndvi_start, ndvi_end, clim_start, clim_end):
    rows = []
    for year in range(2001, 2025):
        ndvi_img = ndvi.filterDate(f'{year}-{ndvi_start}', f'{year}-{ndvi_end}').max().multiply(0.0001).updateMask(mask)
        ndvi_value = ndvi_img.reduceRegion(ee.Reducer.mean(), region, 1000).get('NDVI').getInfo()

        climate = era5.filterDate(f'{year}-{clim_start}', f'{year}-{clim_end}')
        temp = climate.select('temperature_2m').mean().subtract(273.15)
        rain = climate.select('total_precipitation_sum').sum().multiply(1000)
        soil = climate.select('volumetric_soil_water_layer_1').mean()
        clim = temp.addBands(rain).addBands(soil).reduceRegion(ee.Reducer.mean(), region, 5000).getInfo()

        rows.append({
            'year': year,
            'ndvi': ndvi_value,
            'temp': clim['temperature_2m'],
            'rain': clim['total_precipitation_sum'],
            'soil': clim['volumetric_soil_water_layer_1'],
        })
        print(year, end=' ')
    return pd.DataFrame(rows)

df_winter = get_data(winter.gt(50), '04-01', '07-01', '04-01', '07-01')
df_winter.to_csv('winter_2001_2024.csv', index=False)

df_maize = get_data(maize.gt(50), '07-01', '09-01', '06-01', '09-01')
df_maize.to_csv('maize_2001_2024.csv', index=False)

df_spring = get_data(spring.gt(50), '06-01', '08-01', '05-01', '08-01')
df_spring.to_csv('spring_2001_2024.csv', index=False)

In [ ]:
crops_corr = pd.DataFrame({
    'Озимые': df_winter[['temp', 'rain', 'soil']].corrwith(df_winter['ndvi']),
    'Яровые': df_spring[['temp', 'rain', 'soil']].corrwith(df_spring['ndvi']),
    'Кукуруза': df_maize[['temp', 'rain', 'soil']].corrwith(df_maize['ndvi']),
})
crops_corr.plot(kind='bar', figsize=(8, 4))
plt.axhline(0, color='black', linewidth=0.8)
plt.title('Корреляция NDVI с климатом: озимые vs кукуруза vs яровые')
plt.ylabel('r Пирсона')
plt.xticks(rotation=0)
plt.show()

In [ ]:
df = pd.read_csv('winter_2001_2024.csv')
print(df.describe())

In [ ]:
columns = ['ndvi', 'temp', 'rain', 'soil']
titles = ['NDVI озимых (пик апр–июн)', 'Температура апр–июн, °C', 'Осадки апр–июн, мм', 'Влажность почвы']
colors = ['green', 'red', 'blue', 'brown']

fig, axes = plt.subplots(4, 1, figsize=(10, 10))
plt.subplots_adjust(hspace=0.5)
for ax, col, title, color in zip(axes, columns, titles, colors):
    ax.plot(df['year'], df[col], marker='o', color=color)
    ax.set_title(title)
    ax.grid(alpha=0.3)
plt.show()

In [ ]:
corr = df[columns].corr()
sns.heatmap(corr, annot=True, cmap='RdBu', vmin=-1, vmax=1)
plt.title('Корреляция климата и NDVI')
plt.show()
# +1: когда одно растет другое тоже
# 0: связи нет
# -1: когда одно растет другое падает

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(15, 4))
for ax, col in zip(axes, ['temp', 'rain', 'soil']):
    sns.regplot(x=df[col], y=df['ndvi'], ax=ax)
    r, p = stats.pearsonr(df[col], df['ndvi'])
    ax.set_title(f'{col}: r = {r:.2f}, p = {p:.3f}')
plt.tight_layout()

for ax, col in zip(axes[3:], ['temp', 'soil']):
    sns.regplot(x=df[col], y=df['rain'], ax=ax)
    r, p = stats.pearsonr(df[col], df['rain'])
    ax.set_title(f'{col}: r = {r:.2f}, p = {p:.3f}')
plt.show()

In [ ]:
df['anomaly'] = df['ndvi'] - df['ndvi'].mean()
colors = ['red' if x < 0 else 'green' for x in df['anomaly']]

plt.figure(figsize=(10, 4))
plt.bar(df['year'], df['anomaly'], color=colors)
plt.axhline(0, color='black', linewidth=0.8)
plt.title('Отклонение NDVI от среднего по годам')
plt.show()

print(df.sort_values('ndvi').head(3))

In [ ]:
slope, intercept, r, p, se = stats.linregress(df['year'], df['temp'])

plt.plot(df['year'], df['temp'], marker='o', color='red', label='температура')
plt.plot(df['year'], intercept + slope * df['year'], '--', color='black', label='тренд')
plt.legend()
plt.title(f'Тренд: {slope*10:+.2f} °C за 10 лет (p = {p:.3f})')
plt.show()